# Joshi Part 7: Payoffs & Monte Carlo (Rust)

Rust implementation using the `barter-joshi` crate.

**Kernel**: evcxr (Rust Jupyter kernel)

In [ ]:
:dep barter-joshi = { path = "../../barter-joshi" }
:dep rand = "0.9"

## 1. PayOff Trait and Types

In [ ]:
use barter_joshi::payoff::*;

let call = Call::new(100.0);
let put = Put::new(100.0);
let dc = DigitalCall::new(100.0);
let dd = DoubleDigital::new(90.0, 110.0);
let straddle = Straddle::new(100.0);

for spot in [90.0, 100.0, 110.0] {
    println!("S={spot:.0}: Call={:.1} Put={:.1} Digital={:.1} DblDigital={:.1} Straddle={:.1}",
        call.payoff(spot), put.payoff(spot), dc.payoff(spot), dd.payoff(spot), straddle.payoff(spot));
}

## 2. Dynamic Dispatch (Joshi's PayOffBridge)

In [ ]:
let payoffs: Vec<Box<dyn PayOff>> = vec![
    Box::new(Call::new(100.0)),
    Box::new(Put::new(100.0)),
    Box::new(DigitalCall::new(100.0)),
    Box::new(Straddle::new(100.0)),
];

let spot = 105.0;
println!("Spot = {spot}");
for (i, p) in payoffs.iter().enumerate() {
    println!("  PayOff[{i}] = {:.2}", p.payoff(spot));
}

## 3. Simple Monte Carlo

In [ ]:
use barter_joshi::monte_carlo::*;

let call = Call::new(100.0);
let mut rng = rand::rng();

let result = simple_monte_carlo(&call, 100.0, 0.05, 0.2, 1.0, 200_000, &mut rng);
println!("{result}");
println!("BS exact: 10.4506");

## 4. Convergence Table

In [ ]:
let call = Call::new(100.0);
let mut rng = rand::rng();

let (result, convergence) = monte_carlo_with_convergence(
    &call, 100.0, 0.05, 0.2, 1.0, 1 << 18, &mut rng,
);

convergence.print();
println!("\nFinal: {result}");

## 5. Anti-thetic Variance Reduction

In [ ]:
let call = Call::new(100.0);
let mut rng1 = rand::rng();
let mut rng2 = rand::rng();

let basic = simple_monte_carlo(&call, 100.0, 0.05, 0.2, 1.0, 100_000, &mut rng1);
let anti = monte_carlo_antithetic(&call, 100.0, 0.05, 0.2, 1.0, 100_000, &mut rng2);

println!("Basic:      {basic}");
println!("Antithetic: {anti}");
println!("Variance reduction: {:.1}x", basic.std_error / anti.std_error);